This notebook demonstrates seed averaging to improve model stability and reduce variance. Multiple models trained with different random seeds are averaged to produce more robust and reliable predictions compared to single-seed training.

## Importing training and test data:

In [41]:
import pandas as pd

train_path = '/kaggle/input/competitions/playground-series-s6e2/train.csv'
train_df = pd.read_csv(train_path)

orig_path = "/kaggle/input/datasets/cdeotte/s6e4-original-dataset/Heart_Disease_Prediction.csv"
orig_df = pd.read_csv(orig_path)

test_path = '/kaggle/input/competitions/playground-series-s6e2/test.csv'
test_df = pd.read_csv(test_path)

In [42]:
target = 'Heart Disease' 

#Encoding the target
train_df[target] = train_df[target].map({'Absence':0, 'Presence':1})
orig_df[target] = orig_df[target].map({'Absence':0, 'Presence':1})

#removing redundant features and seperating features and target
X = train_df.drop(columns=['id',target], axis=1)
y = train_df[target]

X_orig = orig_df.drop(columns=[target],axis=1)
y_orig = orig_df[target]

# Removing redundant features
X_test = test_df.drop('id', axis=1)

## Feature engineering:
Trying the best feature engineering methods to create new and more informative features. This section will consist:

* Putting numerical features in Bins (Converting numerical to categorical)
* Groupby categorical features and aggrigate statistics of numerical features
* Frequency Encoding
* Perform Target encoding (more about it later)
* Feature Clustering


We'll create custom transformers using **BaseEstimaator** and **TransformerMixin**, what ScikitLearn does is:
1. Call **fit()** on your transformer
2. Then immediately call **transform()** on that SAME training data, without an explicit call

### Binning transformer:

In [43]:
from sklearn.base import BaseEstimator, TransformerMixin

class Binning(BaseEstimator, TransformerMixin):
    def __init__(self, col_to_bin, num_bins, new_col_name ,labels=None):
        self.col_to_bin = col_to_bin
        self.num_bins = num_bins
        self.labels = labels
        self.new_col_name = new_col_name

    def fit(self, X, y=None):
        X = X.copy()
        _, self.bin_edges = pd.cut(X[self.col_to_bin], bins=self.num_bins, labels=False, retbins=True) #the last attribute ensures that the same bins are used to bin teh test data during transform
        return self

    def transform(self,X):
        X = X.copy() 
        X[self.new_col_name] = pd.cut(X[self.col_to_bin], bins=self.bin_edges, labels=False)
        return X

### GroupMean Encoder:

In [44]:
class GroupMeanEncoder(BaseEstimator, TransformerMixin):
    def __init__(self, groupby_col, agg_col, new_col_name):
        self.groupby_col = groupby_col
        self.agg_col = agg_col
        self.new_col_name = new_col_name

    def fit(self,X,y=None):
        self.means = X.groupby(self.groupby_col,observed=True)[self.agg_col].mean()
        return self

    def transform(self,X):
        X = X.copy()
        X[self.new_col_name] = X[self.groupby_col].map(self.means)
        return X

### Frequency Encoder: 
Frequency Encoding is a way to convert a categorical feature into numbers by replacing each category with how often it appears in the dataset.

In [45]:
class FreqEncoder(BaseEstimator, TransformerMixin):
    def __init__(self, cat_cols, normalize=True, orig_data=X_orig, test_data=X_test):
        self.cat_cols = cat_cols
        self.orig_data = orig_data
        self.test_data = test_data
        self.normalize = normalize
        self.freq_maps = {}

    def fit(self, X, y=None):
        X = X.copy()
        # X = pd.concat([X,self.test_data,self.orig_data], axis=0)
        for col in self.cat_cols:
            self.freq_maps[col] = X[col].value_counts(normalize=self.normalize)
        return self

    def transform(self, X):
        X = X.copy()

        for col in self.cat_cols:
            X[col + '_freq'] = X[col].map(self.freq_maps[col])
            X[col + '_freq'] = X[col + '_freq'].fillna(0) 
        return X

### Target Encoder:
Target Encoding = replacing a categorical value with the mean of the target for that category.

In [46]:
class TargetEncoder(BaseEstimator, TransformerMixin):
    def __init__(self, cat_cols, orig_data=X_orig, orig_labels=y_orig):
        self.cat_cols = cat_cols
        self.orig_data = orig_data
        self.orig_labels = orig_labels
        self.mean_maps = {}
        self.global_mean = None
        
    def fit(self, X, y):
        X = X.copy()
        self.global_mean = y.mean()

        # X = pd.concat([X,self.orig_data],axis=0)
        # y = pd.concat([y, self.orig_labels], axis=0)

        for col in self.cat_cols:
            self.mean_maps[col] = y.groupby(X[col]).mean()
        return self

    def transform(self, X):
        X = X.copy()
        
        for col in self.cat_cols:
            X[col + "_TE"] = X[col].map(self.mean_maps[col])
            X[col + "_TE"] = X[col + "_TE"].fillna(self.global_mean) #handling unseen values
        return X

### Feature Clustering:
Added this to teh preprocessing pipeline, might serve as a good feature

In [47]:
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans

class Clustering(BaseEstimator, TransformerMixin):
    def __init__(self, num_clusters=4, add_distance=True ,random_state=42):
        self.num_clusters = num_clusters
        self.random_state = random_state
        self.add_distance = add_distance

    def fit(self, X, y=None):
        self.scaler = StandardScaler()
        X_scaled = self.scaler.fit_transform(X)

        self.kmeans = KMeans(
            n_clusters = self.num_clusters,
            random_state = self.random_state,
            n_init = 10
        )

        self.kmeans.fit(X_scaled)
        return self

    def transform(self, X):
        X = X.copy()
        X_scaled = self.scaler.transform(X)
        cluster_labels = self.kmeans.predict(X_scaled)
        X["Cluster"] = cluster_labels

        if self.add_distance: #i.e. if add_distance==True
            distances = self.kmeans.transform(X_scaled) #distance of the point from each cluster center
            
            for i in range(self.num_clusters):
                X[f"Clust_dist_{i}"] = distances[:,i] #1 column for distance from each cluster
                
        return X

## Preprocessing pipelines:

In [48]:
cat_cols = ['Sex','Chest pain type','FBS over 120','Exercise angina','EKG results']

In [49]:
from sklearn.pipeline import Pipeline

preprocessor = Pipeline([
    ('Binning', Binning(col_to_bin='Age', num_bins=3, new_col_name='Age_bins')),
    ('GroupMeanEncoder_BP', GroupMeanEncoder(groupby_col='Age_bins', agg_col='BP', new_col_name='X1')),
    ('GroupMeanEncoder_Cholesterol', GroupMeanEncoder(groupby_col='Age_bins', agg_col='Cholesterol', new_col_name='X2')),
    ('GroupMeanEncoder_HR', GroupMeanEncoder(groupby_col='Age_bins', agg_col='Max HR', new_col_name='X3')),
    ('FreqEncoding', FreqEncoder(cat_cols=cat_cols)),
    ('TargetEncoding', TargetEncoder(cat_cols=cat_cols)),
    ('Clustering', Clustering(num_clusters=3, add_distance=False))
])

## Model:

In [50]:
best_params_xgb = {'n_estimators': 2634, 
                   'learning_rate': 0.0468112533159457, 
                   'max_depth': 3, 
                   'min_child_weight': 5, 
                   'gamma': 1.4373368015867447, 
                   'subsample':  0.7758150044556165, 
                   'colsample_bytree': 0.39762653718463103, 
                   'reg_alpha': 0.1260400161073486, 
                   'reg_lambda': 0.00014786267797526253,
                  
                    # 'random_state':42,
                    'eval_metric':"logloss",
                    'tree_method':"hist",
                    'device':"cuda",
                    'verbosity':0}

## XGBoost:

In [51]:
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import roc_auc_score
from xgboost import XGBClassifier
import numpy as np

seeds = [42, 1337, 2024, 999, 555, 777, 888, 1234, 5678, 9876]  
all_oof_preds = []
all_test_preds = []

for seed in seeds:
    print(f"\nRunning seed {seed}")
    
    skf = StratifiedKFold(
        n_splits=5,
        shuffle=True,
        random_state=seed
    )
    
    oof_preds = np.zeros(len(X))
    test_preds = np.zeros(len(X_test))
    fold_scores = []

    for fold, (train_idx, val_idx) in enumerate(skf.split(X, y)):
        
        X_tr, X_val = X.iloc[train_idx], X.iloc[val_idx]
        y_tr, y_val = y.iloc[train_idx], y.iloc[val_idx]
        
        model = XGBClassifier(
            **best_params_xgb,
            random_state=seed
        )
        
        model.fit(X_tr, y_tr)
        
        val_preds = model.predict_proba(X_val)[:, 1]
        oof_preds[val_idx] = val_preds
        
        test_preds += model.predict_proba(X_test)[:, 1] / N_SPLITS
        
        fold_scores.append(roc_auc_score(y_val, val_preds))

    print(f"Seed {seed} Mean AUC:", np.mean(fold_scores))
    
    all_oof_preds.append(oof_preds)
    all_test_preds.append(test_preds)


Running seed 42
Seed 42 Mean AUC: 0.9555918891911841

Running seed 1337
Seed 1337 Mean AUC: 0.9555739802896882

Running seed 2024
Seed 2024 Mean AUC: 0.9555861849800358

Running seed 999
Seed 999 Mean AUC: 0.9556214757705833

Running seed 555
Seed 555 Mean AUC: 0.955590671337107

Running seed 777
Seed 777 Mean AUC: 0.9555687227159908

Running seed 888
Seed 888 Mean AUC: 0.9555859953443221

Running seed 1234
Seed 1234 Mean AUC: 0.9555876929463734

Running seed 5678
Seed 5678 Mean AUC: 0.9555795259120273

Running seed 9876
Seed 9876 Mean AUC: 0.9555742103620103


In [52]:
final_oof = np.mean(all_oof_preds, axis=0)
final_test = np.mean(all_test_preds, axis=0)

print("Final OOF AUC:",
      roc_auc_score(y, final_oof))

Final OOF AUC: 0.9556392667731874


In [53]:
submission = pd.DataFrame({
    'id': test_df['id'],
    target: final_test
})

submission.to_csv('submission.csv', index=False)